---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>
<h1 align="center">Course: Generative and Agentic AI</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-22: Fine-Tuning LLMs - I</h1>

# Learning Agenda of This Notebook

1. How to Get the Maximum out of a Model?
    - Comparison between Prompt Engineering, RAG and Fine-Tuning
    - What is Fine-Tuning?
    - Key Reasons and Use Cases forDo Fine-Tuning
2. Pre-Training or Self-Supervised Learning of LLMs
3. Fine-Tuning of Large Language Models
4. Fine-Tuning Strategies of LLMs
5. Hugging Face Developer Ecosystem and Libraries
6. A Deep Dive into the Working of LoRA (Low-Rank Adaptation)
    - The Problem — Why Full Fine-Tuning is Impractical
    - What is Rank of a Matrix?
    - What is Low-Rank Decomposition?
    - How LoRA Exploits The Low-Rank Decomposition?
    - Why Rank Matters for LoRA?
    - Step by Step Working of LoRA
    - Key LoRA Hyperparameters
    - LoRA Architecture Diagram
    - How LoRA Solves the 112 GB Problem
8. Example Questions to Check your Understanding about LoRA

# <span style='background :lightgreen' >1. How to Get the Maximum out of a Model?</span>

## a. Comparison between Prompt Engineering, RAG and Fine-Tuning

| **Aspect**                   | **Prompt Engineering**                                                                | **Fine-Tuning**                                                                 | **Retrieval-Augmented Generation (RAG)**                                    |
| ---------------------------- | ------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------- | --------------------------------------------------------------------------- |
| **Core Idea / Approach**     | Craft effective prompts or few-shot examples to guide the model’s existing knowledge. | Modify the model’s internal weights using new domain-specific data.             | Retrieve relevant external information at query time and add it to prompts. |
| **Data Needs**               | No extra training data; few examples or clear instructions suffice.                   | Hundreds to thousands of high-quality labeled examples for supervised training. | External data sources (documents, PDFs, DBs) stored in a vector database.   |
| **Technical Complexity**     | Low — mostly creative writing and testing.                                            | High — requires ML/engineering skills, GPU setup, and training pipelines.       | Moderate — needs retrieval pipeline, vector DB, and prompt integration.     |
| **Compute & Cost**           | Minimal — inference only.                                                             | High — requires GPU training, time, and expertise.                              | Moderate — retrieval adds latency; no model retraining required.            |
| **Latency / Response Speed** | Fast — no external lookup.                                                            | Fast — inference is quick once model is trained.                                | Slower — retrieval at runtime adds delay (~30–50% overhead).                |
| **Adaptability**             | Very flexible — prompts can be updated instantly.                                     | Static — behavior changes require retraining.                                   | Flexible — update the knowledge base without retraining the model.          |
| **Knowledge Update Method**  | Rephrase or change prompts.                                                           | Retrain or fine-tune the model with new data.                                   | Add or update documents in the vector store.                                |
| **Accuracy & Consistency**   | Variable — depends heavily on prompt quality.                                         | High for structured tasks; risk of overfitting or forgetting unrelated info.    | High — answers are grounded in retrieved sources.                           |
| **Hallucination Risk**       | Higher — model may generate unsupported facts.                                        | Moderate — limited to training data; can still hallucinate outside dataset.     | Lower — answers cite real sources.                                          |
| **Storage Requirements**     | None.                                                                                 | Large — need to store new model checkpoints.                                    | Requires vector embeddings and a database (FAISS, Chroma, etc.).            |
| **Best Use Cases**           | Quick prototyping, creative tasks, UX experimentation.                                | Domain-specific models, style consistency, structured tasks.                    | QA/chatbots needing current or factual information.                         |
| **Maintenance**              | Low — simply edit or discard prompts.                                                 | High — retraining needed periodically to update knowledge or behavior.          | Medium — manage vector DB and retrieval pipeline; no retraining.            |
| **Example**                  | Instruction tuning: "You are a helpful assistant…"                                    | Training a legal LLM or medical diagnosis model.                                | Chatbot pulling answers from internal wiki, manuals, or policies.           |



<div style="text-align:center;">
    <img src="../images/ce2.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## b. What is Fine-Tuning?

<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Fine-tuning means taking a pretrained/base/foundation model and training it further on a smaller, task-specific dataset, so it adapts to a specific purpose</h2>

<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Modify the model's internal weights using new domain-specific data</h2>

## c. Key Reasons and Use Cases for Fine-Tuning
- **Domain Adaptation:** A general-purpose LLM trained on internet text has broad knowledge but lacks deep expertise in specialized fields. Fine-tuning on domain-specific data (medical literature, legal documents, financial reports, scientific papers) gives the model vocabulary, context, and reasoning patterns of that field that were underrepresented in pretraining.
- **Task Specialization:** General LLMs are jack-of-all-trades. Fine-tuning sharpens them for a specific tasks like question answering, code generation, summarization, translation, and classification. (often outperform much larger general models on that specific task)
- **Teaching a Custom Personality or Style:** Fine-tuning allows injecting a specific tone, communication style, or persona into the model. For example, giving an assistant the teaching personality of *Dr. Muhammad Arif Butt* by training the model on his lectures, notes, and explanations — so it responds the way Dr. Arif would.
- **Instruction / Chat Behavior:** Pretrained base models predict next tokens and they do not naturally follow instructions or hold conversations. Fine-tuning on instruction-response pairs teaches the model to behave like a helpful assistant that answers questions, follows commands, and engages in dialogue.
- **Reducing Hallucinations in Specific Domains:** General LLMs hallucinate facts in specialized domains because they lack reliable grounding. Fine-tuning on verified domain data anchors the model to accurate, domain-specific knowledge and reduces confident wrong answers.
- **Working with Private / Proprietary Data:** Organizations cannot send sensitive internal data (patient records, financial data, classified documents) to external APIs like OpenAI. Fine-tuning allows building a private, locally-hosted model trained on internal data with full data privacy.
- **Cost & Latency Efficiency:** A fine-tuned small model (1B–7B) can match or exceed the performance of a large general model (70B+) on a narrow task. Fine-tuning is how you get GPT-4 level performance on your specific task from a model that runs on a laptop.
- **Classification or Threat Detection:** Fine-tuning a pretrained model on phishing emails, spam campaigns, and social engineering messages so it can automatically flag malicious content.
- **Malware Analysis:** Adapting a pretrained model on datasets of malware source code or binary signatures to classify unknown samples as ransomware, trojans, or worms.
- **Vulnerability Identification:** Fine-tuning a pretrained model on code snippets with known security flaws (like SQL injection or buffer overflows) so it can detect vulnerabilities in new applications.
- **Log Anomaly Detection:** Teaching a model using real network or system logs containing attack traces to recognize unusual or malicious activity patterns in real time.
- **Incident Report Summarization:** Fine-tuning a pretrained model on past SOC (Security Operations Center) reports to automatically summarize, triage, or prioritize security incidents.
- **Removing or Adjusting Guardrails:** Pretrained models from major providers are aligned with strict safety guidelines that may block legitimate research, red-team security testing, or culturally specific content. Researchers and security professionals sometimes fine-tune models to adjust or remove these restrictions for controlled, ethical research environments.

# <span style='background :lightgreen' >2. Pre-Training of LLMs (Self-Supervised Learning)</span>

<h2 align="center"><div class="alert alert-success" style="margin: 20px"><b>Pre-Training</b> means teaching a model EVERYTHING about language (NLU + NLG) from scratch on massive data (terabytes) from random weights</h2>

> 💡 **Pre-Training** is like a person going through ALL of their education from birth to university:
    >- Learning to read → primary school
    >- Learning maths   → secondary school
    >- General knowledge → university
    >- Takes years, and is enormously expensive

### Step 1: Forward Pass/Propagation:
- Forward propagation is the process of passing input values through the neural network layer by layer to produce a predicted output. Each layer takes values from the previous layer and applies three operations: **multiplication** (weights), **addition** (biases), and a non-linear **activation function** — then passes the result to the next layer. This repeats through all subsequent layers until the final layer produces the model's prediction (e.g., probability distribution over the next token).
```
  Input tokens  →  Embedding  →  Transformer Blocks  →  Output logits
  ["The", "capital", "of", "Pakistan", "is"]             [vocab_size]
                                                               ↓
                                                           Softmax
                                                               ↓
                                                      P("Islamabad") = 0.0003  ← very low!
                                                      (random weights know nothing)

```
### Step 2: Loss Calculation:
- The model's predicted output is compared against the actual target tokens to compute a **loss**, a number that quantifies how wrong the prediction was.
- Loss is HIGH when model assigns LOW probability to the correct next token.
- Loss is ZERO when model is perfectly confident about the correct token.
- Common loss functions used in practice:
    - `mse` (Mean Squared Error): Works for regression tasks, i.e.,  predicting a continuous number (e.g., house price). NOT suitable for next-token prediction because: Output is a probability distribution over 50,000 tokens and MSE does not handle probability distributions well
    - `binary_crossentropy`: Works for binary classification (yes/no, spam/not spam). NOT suitable because next-token prediction has 50,000 possible classes, not just 2
    - `categorical_crossentropy`: Used for multi-class classification and next-token prediction in LLMs to classify which of 50,000 vocabulary tokens comes next
```
Loss = -log( P(correct_token) )
     = -log( P("Islamabad") )
     = -log(0.0003)
     = 8.11   ← very high loss (random prediction was terrible)
```
### (iii) Backward Pass/Propagation: 
- Once the loss is computed, we calculate the **derivative of the loss with respect to every weight** in the ANN/model, these derivatives are called **gradients**.
- `∂Loss/∂W  →  "if I increase this weight slightly, does loss go up or down?"`
- This gradient flows BACKWARDS through every transformer block: `Output layer → MLP layers → Attention layers → Embedding layer`
- Every single weight in the model receives a gradient signal, telling it: "change by this much in this direction"

### (iv) Weight Update (Gradient Descent + Adam Optimizer):
- The computed gradients are  then used by an **optimization algorithm** (such as Adam) to update every weight in the network in the direction that **minimizes the loss**:
```
W_new = W_old - learning_rate × gradient
      = W_old - 0.0001 × ∂Loss/∂W
```
- After this update: `P("Islamabad") might go from 0.0003 → 0.0004  (tiny improvement)`
- Repeat this for TRILLIONS of tokens across THOUSANDS of GPU-hours and gradually the weights encode all of human language knowledge
- This is why it is called **gradient descent**, because we descend the loss surface step by step toward the minimum using the gradient as our compass.
- The **learning rate** is a critical hyperparameter that controls the size  of each update step:
    - Too **high** → overshoots the minimum → training diverges or oscillates
    - Too **low** → takes too long to converge → training is impractically slow
    - Just **right** → steadily descends toward the **local minimum** (point of minimum loss) in a reasonable number of steps
- Here is a brief summary of smarter **optimization algorithms** that converge faster and more reliably than basic gradient descent:
<table style="margin-left: 0; margin-right: auto;">
  <thead>
    <tr>
      <th>Optimizer</th>
      <th>Key Idea</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>SGD</b> (Stochastic Gradient Descent)</td>
      <td>Updates weights using <b>one random training example</b> at a time instead of full dataset. It is fast and memory efficient but very noisy updates that oscillate around the minimum</td>
    </tr>
    <tr>
      <td><b>BGD</b> (Batch Gradient Descent)</td>
      <td>Update weights using the <b>entire dataset</b> of data. It is very stable and accurate but very slow</td>
    </tr>
    <tr>
      <td><b>MBGD</b> (Mini-Batch Gradient Descent)</td>
      <td>Update weights using <b>small batches</b> of data(typically 32–256 examples). It is a practical compromise between SGD and BGD</td>
    </tr>
    <tr>
      <td><b>SGD + Momentum</b></td>
      <td>Adds a <b>running average of past gradients</b> (like a ball rolling downhill that builds up speed) to SGD updates — smooths out oscillations and accelerates convergence in consistent directions</td>
    </tr>
    <tr>
      <td><b>AdaGrad</b> (Adaptive Gradient)</td>
      <td>Adapts the learning rate <b>per parameter</b> by dividing by the accumulated sum of past squared gradients. Gives larger updates for rare features, smaller for frequent ones</td>
    </tr>
    <tr>
      <td><b>RMSProp</b> (Root Mean Square Propagation)</td>
      <td>Fixes AdaGrad's vanishing learning rate problem by using an <b>exponentially weighted moving average</b> of squared gradients instead of accumulating all past squared gradients</td>
    </tr>
    <tr>
      <td><b>Adam</b> (Adaptive Moment Estimation)</td>
      <td>Combines <b>SGD + Momentum</b> (tracks the direction of past gradients — 1st moment) and <b>RMSprop</b> (tracks the magnitude of past gradients — 2nd moment). Most widely used optimizer for training and fine-tuning LLMs</td>
    </tr>
    <tr>
      <td><b>AdamW</b> (Adam + Weight Decay)</td>
      <td> <b>Identical to Adam but fixes a subtle bug where Adam's L2 regularization was applied incorrectly</td>
    </tr>
      
  </tbody>
</table>

<br><br>

> 💡 **Self-Supervised Learning:** In LLM pre-training, a/m four-step cycle is repeated over **trillions of tokens** from massive text corpora (web pages, books, code, Wikipedia etc). No labeling is required as the **next token itself  serves as the label**. Labels are AUTOMATICALLY generated FROM THE DATA ITSELF. No human annotation required. After seeing **trillions of tokens**, the weights gradually encode:
    >- Grammar and syntax
    >- World knowledge and facts
    >- Reasoning patterns
    >- Code structure
    >- Multilingual understanding

# <span style='background :lightgreen' >3. Fine-Tuning of Large Language Models (Supervised Learning)</span>
<h2 align="center"><div class="alert alert-success" style="margin: 20px"><b>Fine-Tuning</b> means teaching a model a SPECIFIC skill or behaviour on top of pre-trained weights using small, curated, labeled data (megabytes to gigabytes)</h2>

> 💡 **Fine-Tuning** is like a university graduate doing a short specialist course or internship:
    >- Learns a specific skill quickly (weeks not years)
    >- Builds on existing knowledge and need not to start from scratch
    >- Much cheaper and faster
    >- The core knowledge (language) stays intact
    >- Only specialist behavior changes

### Step 1: Forward Pass/Propagation:
- The forward pass works identically to pre-training; input tokens flow layer by layer through embeddings, transformer blocks, and output logits.
- The **critical difference** is the starting point; instead of random weights, fine-tuning begins from pre-trained weights that already encode rich language knowledge. The model is already a competent language user — it just needs to be steered.
- The **input format** is also different. Fine-tuning data is mostly structured as **instruction–response pairs** rather than **raw text**:

```
Input tokens  →  Embedding  →  Transformer Blocks  →  Output logits
["Summarise", "this", "contract", "in", "simple", "English:"]    [vocab_size]
                                                                       ↓
                                                                   Softmax
                                                                       ↓
                                                        P("This") = 0.61  ← already reasonable!
                                                        (pre-trained weights know language well)
```

> 🔑 **Key Difference:** Pre-training starts from **random weights** (P ≈ 0.0003). Fine-tuning starts from **pre-trained weights** (P ≈ 0.40–0.80), so the model is already in a good region of the loss surface before a single gradient step is taken.

### Step 2: Loss Calculation:
- The loss function mechanics are the same; predicted token probabilities are compared against target tokens.
- The **critical difference** is *where loss is computed*. During fine-tuning on instruction data, loss is calculated **only on the response/completion tokens**, NOT on the instruction/prompt tokens. The model is penalised only for getting the *answer* wrong, not the *question* wrong.
- Because the model already understands language well, the loss starts **much lower** than in pre-training and converges in far fewer steps.

```
Prompt  (masked — no loss computed here):
   "Classify the sentiment of this review: 'The product broke in a week.'"

Completion (loss is computed here):
Loss = -log( P("Negative") )
     = -log(0.61)
     = 0.49   ← already low loss (pre-trained weights gave a strong head-start)
```

> 🔑 **Key Difference:** Pre-training computes loss over **every token** in the corpus. Fine-tuning masks the prompt and computes loss **only over target response tokens**, so gradients reflect task-specific correctness, not general next-token prediction.


### Step 3: Backward Pass/Propagation:
- The backward pass is **mechanically identical** — gradients of the loss flow backwards through every transformer block, from output layer back to embedding layer.
- The **critical differences** are **magnitude and scope**:
  - Gradients are **much smaller** because the loss is already low — tiny nudges are needed, not large corrections.
  - In **full fine-tuning**, all weights receive gradient updates (same as pre-training).
  - In **parameter-efficient fine-tuning (PEFT)** methods such as **LoRA**, gradients are computed for all weights but updates are applied **only to a small set of adapter parameters** — the majority of pre-trained weights are frozen, protecting the general knowledge acquired during pre-training.

```
Output layer → MLP layers → Attention layers → Embedding layer
      ↓
Gradients are SMALL (model is already close to correct)
      ↓
In LoRA: only low-rank adapter matrices A and B are updated
         original weight matrix W remains FROZEN
```

> 🔑 **Key Difference:** Pre-training sends **large gradients** across **all weights** to build knowledge from scratch. Fine-tuning sends **small, targeted gradients** — either to all weights (full fine-tuning) or to a tiny fraction of adapter weights (PEFT/LoRA) — to refine behaviour without erasing prior knowledge.


### Step 4: Weight Update (Gradient Descent + Adam Optimizer):
- The update rule is **the same formula** as pre-training, but the **hyperparameters are set very differently**:

```
W_new = W_old - learning_rate × gradient
      = W_old - 0.000002 × ∂Loss/∂W   ← learning rate is 10×–100× SMALLER than pre-training
```

- After this update: `P("Negative") might go from 0.61 → 0.63  (gentle refinement)`
- Repeat this for **thousands to millions** of examples over **hours to days** on far fewer GPUs than pre-training required.
- A **much smaller learning rate** is essential during fine-tuning:
    - Too **high**   → **catastrophic forgetting** — the model overwrites its pre-trained general knowledge with narrow task-specific behaviour
    - Too **low**    → the model fails to adapt meaningfully to the new task
    - Just **right** → the model **specialises** on the target task while **retaining** its broad pre-trained capabilities

> 🔑 **Key Difference:** Pre-training uses a **larger learning rate** over **trillions of tokens** to carve knowledge into weights from scratch. Fine-tuning uses a **much smaller learning rate** over **thousands to millions of curated examples** to gently steer behaviour — preserving the pre-trained foundation while building task-specific expertise on top of it.

<br><br>

> 💡 **Supervised Learning:** In fine-tuning, the four-step cycle is repeated over a **small, curated, labeled dataset** of instruction–response pairs. Unlike pre-training, labels are **NOT automatically generated from data** — they are **human-authored** (e.g., via human annotators writing ideal responses, or via templates). After seeing **thousands to millions of examples**, the weights are refined to encode:
>    - Instruction-following behaviour
>    - Domain-specific knowledge and terminology
>    - A desired tone, format, or response style
>    - Safety and alignment constraints (in RLHF-based fine-tuning)
>    - Task-specific reasoning patterns (e.g., medical, legal, coding)
```

# <span style='background :lightgreen' >4. Fine-Tuning Strategies of LLMs</span>

<pre>
                         <b>LLM Fine-Tuning Strategies</b>
┌──────────────────────────────────────────────────────────────────────┐
│                         <b>Base Pretrained LLM</b>                          │
└──────────────────────────────────────────────────────────────────────┘
                                   │
                                   ▼
┌──────────────────────────────────────────────────────────────────────┐
│         <b>(a) Continued Pretraining / Domain Adaptation </b>               │
│          Self-Supervised Learning (Next-Token Prediction)            │
│          Example: Legal, Medical, Finance domain corpora             │
└──────────────────────────────────────────────────────────────────────┘
                                   │
                                   ▼
┌──────────────────────────────────────────────────────────────────────┐
│          <b>(b) Supervised Fine-Tuning (SFT) </b>                           │
│          Train using labeled input → output examples                 │
│                                                                      │
│   ┌──────────────────────────────────────────────────────────────┐   │
│   │ (i)  Full Fine-Tuning                                        │   │
│   │      Update ALL model parameters                             │   │
│   └──────────────────────────────────────────────────────────────┘   │
│                                                                      │
│   ┌──────────────────────────────────────────────────────────────┐   │
│   │ (ii) Partial Fine-Tuning                                     │   │
│   │      Freeze most layers, train only some layers              │   │
│   └──────────────────────────────────────────────────────────────┘   │
│                                                                      │
│   ┌──────────────────────────────────────────────────────────────┐   │
│   │ (iii) Parameter-Efficient Fine-Tuning (PEFT)                 │   │
│   │       Freeze base model, train small adapter modules         │   │
│   │       Examples: LoRA, QLoRA, DoRA, IA3, Adapters, BitFit     │   │
│   └──────────────────────────────────────────────────────────────┘   │
│                                                                      │
│   ┌──────────────────────────────────────────────────────────────┐   │
│   │ (iv) Instruction Fine-Tuning                                 │   │
│   │      Train model to follow natural language instructions     │   │
│   │      Example: Q&A, task instructions, chat datasets          │   │
│   └──────────────────────────────────────────────────────────────┘   │
└──────────────────────────────────────────────────────────────────────┘
                                   │
                                   ▼
┌──────────────────────────────────────────────────────────────────────┐
│      <b> (c) Alignment via Preference Learning (Post-SFT)</b>               │
│           Improve helpfulness, safety, and response quality          │
│                                                                      │
│                   RLHF,  DPO, RLAIF, GRPO                            │
└──────────────────────────────────────────────────────────────────────┘
</pre>

## a. Continued Pretraining / Domain Adaptation (Self-Supervised)
- Used when adapting a model to a new domain *without* labeled data
- The model continues learning via **next-token prediction** on raw domain-specific text (e.g., legal documents, financial reports, medical literature)
- Called "self-supervised" because the data itself serves as the label — no human annotation required
> **Note:** "Self-supervised" refers to the *training objective* (predicting next tokens), not merely the absence of labels

### Datasets
 The following datasets are used in **self-supervised learning** having raw text with no target labels. The model learns by predicting the next token on massive corpora. Use these when you want to adapt a model to a new domain.

| Dataset | Description | Link |
|---|---|---|
| **Common Crawl** | Petabyte-scale web crawl data; the backbone of most LLM pretraining | 🔗 [common-crawl-sample](https://huggingface.co/datasets/agentlans/common-crawl-sample) |
| **C4** | Cleaned version of Common Crawl by Google; used to train T5 | 🔗 [allenai/c4](https://huggingface.co/datasets/allenai/c4) |
| **Wikipedia** | Clean, factual encyclopedic text in 100+ languages | 🔗 [wikimedia/wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia) |
| **Project Gutenberg** | 70,000+ public domain books — great for literary/general language modeling | 🔗 [sedthh/gutenberg_english](https://huggingface.co/datasets/sedthh/gutenberg_english) |
| **ArXiv** | Scientific papers from physics, math, CS, biology etc. | 🔗 [ArXiv Dataset on Kaggle](https://www.kaggle.com/datasets/Cornell-University/arxiv) |
| **GitHub Code** | Large-scale source code across many programming languages | 🔗 [codeparrot/github-code](https://huggingface.co/datasets/codeparrot/github-code) |
| **StackExchange** | Q&A pairs from Stack Overflow and 170+ sister sites | 🔗 [HuggingFaceH4/stack-exchange-preferences](https://huggingface.co/datasets/HuggingFaceH4/stack-exchange-preferences) |
| **The Pile** | 800GB diverse text corpus by EleutherAI combining 22 high-quality sources | 🔗 [EleutherAI/pile](https://huggingface.co/datasets/EleutherAI/pile) |
| **RedPajama** | Open reproduction of LLaMA pretraining data | 🔗 [togethercomputer/RedPajama-Data-1T](https://huggingface.co/datasets/togethercomputer/RedPajama-Data-1T) |


## b. Supervised Fine-Tuning / SFT (with labeled input-output pairs)
### (i) Full Fine-Tuning
- All model weights and parameters are updated during training
- Retweaking billions of parameters is compute/memory expensive and time consuming
- Requires the same order of memory as pretraining — typically impractical for models 7B+
- Produces the best possible task performance but at the highest cost
### (ii) Partial Fine-Tuning (Layer Freezing / Transfer Learning)
- Instead of updating all parameters, most layers are **frozen** (weights locked) and only a selected subset of layers is trained
- Common strategy is to freeze the early layers (which capture general language knowledge) and fine-tune only the **top N transformer blocks** or the **LM Head** (final classification/generation layers)
- Much cheaper than full fine-tuning but less flexible than PEFT
### (iii) Parameter Efficient Fine-Tuning (PEFT)
- In PEFT, **all original model parameters are frozen** — we do not modify the pretrained weights at all
- Instead, a small number of **new trainable parameters** are added to or around the model, and only these are updated during fine-tuning
- Achieves performance close to full fine-tuning at a fraction of the compute and memory cost

**Common PEFT Techniques:**
- **[LoRA (2021)](https://arxiv.org/abs/2106.09685)** *(Low-Rank Adaptation)*: Injects small trainable low-rank matrices alongside the frozen weight matrices of attention layers. Instead of updating the full weight matrix W, it learns two small matrices A and B whose product **ΔW = B×A** approximates the weight update — keeping the base model completely frozen
- **[BitFit (2022)](https://arxiv.org/abs/2106.10199)**:  Trains **only the bias terms** of the existing frozen weight matrices. Despite updating less than 0.1% of total parameters it achieves surprisingly competitive results on many NLP benchmarks — making it the most lightweight PEFT technique of all
- **[IA3 (2022)](https://arxiv.org/abs/2205.05638)** *(Infused Adapter by Inhibiting and Amplifying Inner Activations)*: Introduces three small learned **scaling vectors** that rescale the keys, values and feed-forward activations inside each transformer layer.  It is more parameter-efficient than LoRA as it adds virtually no new parameters and use the learned scalars that amplify or suppress existing activations
- **[QLoRA (2023)](https://arxiv.org/abs/2305.14314)** *(Quantized LoRA)*:  Memory-efficient variant of LoRA that quantizes the frozen base model weights to **4-bit NF4 precision** to dramatically reduce memory usage. It then applies standard LoRA trainable adapters on top in full precision, making it possible to fine-tune 7B+ parameter models on a single consumer GPU
- **[DoRA (2024)](https://arxiv.org/abs/2402.09353)** *(Weight-Decomposed LoRA)*: Improved variant of LoRA that decomposes each pre-trained weight matrix into **magnitude** (how large) and **direction** (which way) components. It then applies LoRA only to the *direction component*, resulting in more stable training and better performance than standard LoRA especially at low ranks
- **[Adapters (2024)](https://arxiv.org/abs/2405.05493)**: Inserts small **bottleneck feed-forward modules** inside each transformer block between the attention sublayer and the FFN sublayer. Each adapter projects down to a lower dimension applies a non-linearity then projects back up. Only these adapter weights are trained while the entire base model remains frozen.
    
    
### (iv) Instruction Fine-Tuning
- A specialized form of SFT that teaches the model to follow instructions and behave like a helpful assistant
- The pretrained base model knows language but does not naturally respond to instructions;  instruction fine-tuning bridges this gap
- Training data includes:
    - Task demonstrations
    - Question-Answer pairs
    - Multi-turn conversations

## Datasets
- Following datasets are used in **Supervised Fine-Tuning (SFT)** and are formatted as instruction-response pairs or conversations. These teach the model to follow instructions and behave like an assistant.

| Dataset | Description | Link |
|---|---|---|
| **Alpaca Cleaned** | 52K instruction-following examples cleaned from Stanford Alpaca dataset | 🔗 [yahma/alpaca-cleaned](https://huggingface.co/datasets/yahma/alpaca-cleaned) |
| **OpenAssistant (OASST2)** | High-quality human-written multi-turn conversations and ranked responses | 🔗 [OpenAssistant/oasst2](https://huggingface.co/datasets/OpenAssistant/oasst2) |
| **Dolly 15K** | 15K human-generated instruction-response pairs by Databricks employees | 🔗 [databricks/databricks-dolly-15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k) |
| **UltraChat** | Large-scale multi-turn chat dataset with diverse topics | 🔗 [HuggingFaceH4/ultrachat_200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) |
| **ShareGPT** | Real user-ChatGPT conversations shared publicly | 🔗 [anon8231489123/ShareGPT_Vicuna_unfiltered](https://huggingface.co/datasets/anon8231489123/ShareGPT_Vicuna_unfiltered) |
| **SmolTalk** | High-quality SFT dataset used to train Hugging Face's SmolLM2 models | 🔗 [HuggingFaceTB/smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) |
| **Tulu 3** | State-of-the-art SFT + preference dataset by AllenAI (2024) | 🔗 [allenai/tulu-3-sft-mixture](https://huggingface.co/datasets/allenai/tulu-3-sft-mixture) |
| **arifbutt_dataset** | Personal public information about Dr. Muhammad Arif Butt | 🔗 [arif-butt/arifbutt_dataset](https://huggingface.co/datasets/arif-butt/arifbutt_dataset) |

## c. Alignment via Preference Learning (Post-SFT Stage)
<h2 align="center"><div class="alert alert-success" style="margin: 20px">SFT teaches the model HOW to answer. Alignment teaches the model WHICH answers are better.</h2>

- Applied **after Supervised Fine-Tuning (SFT)** to improve helpfulness, safety, tone, and reliability
- At this stage the model already follows instructions; alignment teaches it **which responses humans prefer**
- Training uses **preference signals** (better vs worse responses) instead of direct ground-truth answers

**Common Alignment Techniques:**

- **[RLHF (2022)](https://arxiv.org/abs/2203.02155) Reinforcement Learning from Human Feedback:**  Human annotators rank multiple model responses. A **reward model** is trained on these rankings, and the LLM is optimized using **[PPO](https://arxiv.org/pdf/1707.06347)** to maximize the reward signal. This was the alignment method used in early systems like ChatGPT.
- **[DPO (2024)](https://arxiv.org/abs/2305.18290)  Direct Preference Optimization:**  A simpler alternative to RLHF that **directly trains the model on (chosen, rejected) response pairs** without requiring a reward model or PPO training loop. Easier, more stable, and widely used in modern LLM alignment.
- **[RLAIF (2024)](https://arxiv.org/abs/2309.00267) Reinforcement Learning from AI Feedback:**   Instead of human annotators, a **strong AI model generates preference judgments**. This greatly reduces labeling cost and enables large-scale alignment.
- **[GRPO (2024)](https://arxiv.org/pdf/2402.03300)  Group Relative Policy Optimization:**  Used in reasoning models such as DeepSeek-R1. The model generates **multiple candidate answers**, which are compared against each other using relative rewards, eliminating the need for a separate value/critic model.
- **[Constitutional AI (2022)](https://arxiv.org/abs/2212.08073):**  A framework introduced by Anthropic where the model follows a **set of predefined principles ("constitution")**. AI feedback is then used to critique and revise responses according to these rules, enabling scalable alignment with minimal human labeling.


### Datasets for Alignment Learning 
> Used in the **alignment stage** (Post-SFT) — formatted as **(chosen, rejected)** response pairs or scored responses. These teach the model which responses are more helpful, safe, and aligned with human values.

| Dataset | Description | Link |
|---|---|---|
| **Descriptiveness Sentiment (TRL)** | Sample preference dataset by Hugging Face TRL team; great for learning DPO/RLHF | 🔗 [trl-internal-testing/descriptiveness-sentiment-trl-style](https://huggingface.co/datasets/trl-internal-testing/descriptiveness-sentiment-trl-style) |
| **UltraFeedback** | 64K instructions with model responses rated by GPT-4; widely used for DPO | 🔗 [HuggingFaceH4/ultrafeedback_binarized](https://huggingface.co/datasets/HuggingFaceH4/ultrafeedback_binarized) |
| **Anthropic HH-RLHF** | Human preference data for helpfulness and harmlessness by Anthropic | 🔗 [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf) |
| **OpenAssistant Reward** | Human-ranked assistant responses for reward model training | 🔗 [OpenAssistant/oasst1](https://huggingface.co/datasets/OpenAssistant/oasst1) |
| **Orca DPO Pairs** | High-quality DPO pairs derived from the Orca dataset | 🔗 [Intel/orca_dpo_pairs](https://huggingface.co/datasets/Intel/orca_dpo_pairs) |
| **Capybara DPO** | Diverse multi-turn preference dataset for DPO training | 🔗 [argilla/distilabel-capybara-dpo-7k-binarized](https://huggingface.co/datasets/argilla/distilabel-capybara-dpo-7k-binarized) |
| **Helpsteer2** | High-quality preference dataset by NVIDIA with fine-grained attribute ratings | 🔗 [nvidia/HelpSteer2](https://huggingface.co/datasets/nvidia/HelpSteer2) |

### Datasets for Chain-of-Thought / GRPO Training
> Used for training models to **think step-by-step** and solve complex problems. These are increasingly important with the rise of reasoning models like DeepSeek-R1 and OpenAI o1. Often used with **GRPO** (Group Relative Policy Optimization) or other RL-based fine-tuning.

| Dataset | Description | Link |
|---|---|---|
| **GSM8K** | 8.5K grade school math word problems requiring multi-step reasoning; standard reasoning benchmark | 🔗 [openai/gsm8k](https://huggingface.co/datasets/openai/gsm8k) |
| **MLX GRPO Dataset** | Dataset specifically formatted for GRPO-based reasoning fine-tuning experiments | 🔗 [Goastro/mlx-grpo-dataset](https://huggingface.co/datasets/Goastro/mlx-grpo-dataset) |
| **NuminaMath** | Competition-level math problems with step-by-step solutions; used in DeepSeek-R1 training pipeline | 🔗 [AI-MO/NuminaMath-CoT](https://huggingface.co/datasets/AI-MO/NuminaMath-CoT) |
| **OpenR1 Math** | Open reproduction of DeepSeek-R1 math reasoning data | 🔗 [open-r1/OpenR1-Math-220k](https://huggingface.co/datasets/open-r1/OpenR1-Math-220k) |
| **Magpie Reasoning** | Large-scale reasoning traces generated by frontier models | 🔗 [Magpie-Align/Magpie-Reasoning-V2-250K-CoT-Deepseek-R1-Llama-70B](https://huggingface.co/datasets/Magpie-Align/Magpie-Reasoning-V2-250K-CoT-Deepseek-R1-Llama-70B) |
| **ARC Challenge** | AI2 Reasoning Challenge — science questions requiring multi-step reasoning | 🔗 [allenai/ai2_arc](https://huggingface.co/datasets/allenai/ai2_arc) |

# <span style='background :lightgreen' >5. Hugging Face Developer Ecosystem and Libraries</span>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The Hugging Face ecosystem consists of multiple libraries that work together to make training, fine-tuning, and deploying AI models easier.</h3>

<pre>
┌─────────────────────────────────────────────────────────────────┐
│                        <b>unsloth</b>                                  │
│   2-5× faster kernels, 60% less VRAM, GGUF export               │
│      (Accelerates training built on Transformers + PEFT + TRL)  │
├────────────────────────────┬────────────────────────────────────┤
│          <b>peft</b>              │          <b>trl</b>                       │
│  LoRA/QLoRA/DoRA           │  Alignment & RLHF training         │
│  Freeze base model         │  SFT, DPO, PPO, GRPO, Reward models│
│  Train only adapters       │  Built on Transformers + PEFT      │
├────────────────────────────┴────────────────────────────────────┤
│                    <b>transformers</b>                                 │
│ Hub Integration, Models, Tokenizers, Configs, Pipelines, Trainer│
│         updates ALL model weights during fine-tuning            │
├─────────────────────────────────────────────────────────────────┤
│              <b>PyTorch / TensorFlow / Keras / JAX</b>                 │
│    (Core DL libraries on which above libraries are built)       │
└─────────────────────────────────────────────────────────────────┘
</pre>



>- **PyTorch** is a flexible, pythonic deep learning framework widely used for research and LLM development.
>- **TensorFlow** is a production-oriented ML/DL framework with strong deployment ecosystem.
>- **Keras** is a high-level user-friendly API for building models, now integrated with TensorFlow.
>- **JAX** (Just After eXecution) is a ML framework that provides fast array computing with automatic differentiation and XLA compilation.

## a. **[`tramsformers` (2018)](https://arxiv.org/pdf/1910.03771)**: Core Library

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">The single most important library to understand is <b>Transformers</b>, every other library in this echosystem (peft, trl, unsloth) is built on top of it.</h3>

<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Hugging Face Transformers is a library that provides tools for <b>using, loading, fine-tuning,</b> and <b>pre-training</b> open-source AI models with a unified API that works seamlessly with Meta's PyTorch and Google's TensorFlow/Keras and JAX.</h3>

- **What it is:** The core Hugging Face library that provides pre-built model architectures, tokenizers, and training utilities for 200+ model families (Llama, Qwen, Mistral, Phi, GPT-2 etc.)
- **What it does:**
    - Load any pretrained model from Hugging Face Hub in one line
    - Tokenize text into model-ready inputs
    - Run inference and training via `Trainer` class
    - Handle model configurations, saving, and pushing to Hub
- **Example(s):**
```python
# HF Hub
from huggingface_hub import login, HfApi
login(token=hf_token)                        
client = HfApi()                             
models_gen = client.list_models(author="arif-butt") 
# Pipeline
from transformers import pipeline
pipe = pipeline("text-generation")
print(pipe("Which is the highest mountain peak in Pakistan?")
# Models
from transformers import AutoModelForCausalLM
model     = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# Tokenizers
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# Config
from transformers import AutoConfig
config = AutoConfig.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# Trainer
from transformers import Trainer, TrainingArguments
training_args = TrainingArguments(...)
trainer = Trainer(
                model         = model,
                args          = training_args,
                train_dataset = train_dataset,
                eval_dataset  = eval_dataset,
                )
trainer.train()
```
- 🔗 [Documentation](https://huggingface.co/docs/transformers) | 📄 [Paper](https://arxiv.org/abs/1910.03771)


## b. **[`peft` (2021)](https://huggingface.co/docs/peft/index)** (Parameter Efficient Fine-Tuning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>peft</b> is a HF library used to fine-tune transformer-based models by injecting small trainable adapter modules while keeping the massive base model weights completely <b>frozen</b>. It is designed exclusively for <b>Supervised Fine-Tuning (SFT)</b> and cannot perform pre-training, continued pre-training, or alignment on its own.</div></h3>

- **Problem it solves:** `transformers`'s built-in `Trainer` updates ALL model weights during fine-tuning — requiring enormous GPU memory (112 GB for 7B model). `peft` reduces this to a tiny fraction by only training small adapter matrices
- **What it is:** Hugging Face library that implements all PEFT techniques — LoRA, QLoRA, Adapters, IA³, DoRA etc.
- **What it does:**
    - Wraps any `transformers` model with LoRA adapter layers in a few lines
    - Freezes base model weights automatically — only adapter weights are trained
    - Saves only the small adapter weights (MBs) instead of the full model (GBs)
    - Supports merging adapters back into base model via `merge_and_unload()`
- **When to use:** Whenever you want to fine-tune efficiently without updating all model weights — i.e., almost always. 
- **Sits on top of:** `transformers` — cannot work without it
- - **Important Links:**
    - 🔗 [Documentation](https://huggingface.co/docs/peft)
    - 📄 [LoRA Paper](https://arxiv.org/abs/2106.09685)
    - [GitHub](https://github.com/huggingface/peft)
- **Example PEFT Pipeline (Manual Adapter Injection):**
```python
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

model = AutoModelForCausalLM.from_pretrained(...)  # Load the base model pretrained weights using HuggingFace Transformers
lora_config = LoraConfig(...)                      # Define the LoRA adapter architecture specifying rank, lora_alpha, target_modules, droptout, task_type
model = get_peft_model(model, lora_config)         # Inject LoRA adapters into the model, which will freeze all base model weights, and insert LoRA A and B matrices into target layers, wraps the model in PEFTModel wrapper
trainer = Trainer(...)                             # Create the HuggingFace Trainer
trainer.train()                                    # Follows the standard transformer training loop and only LoRA parameters receive gradients
```


## c. **[`trl` (2022)](https://arxiv.org/abs/2204.05862)** (Transformer Reinforcement Learning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>trl</b> is a HF library that provides complete training pipelines for every <b>post-pretraining stage</b> of LLM development (SFT, DPO, GRPO, PPO, RLHF and reward modeling) built on top of transformers and peft. (pre-training is outside its scope)</div></h3>

- **Problem it solves:** `transformers`'s `Trainer` is a general-purpose training loop as it has no concept of instruction masking, preference pairs, reward models, or policy optimization. `trl` adds all of these on top as ready-to-use trainers
- **What it is:** Hugging Face library that provides ready-to-use trainers for every stage of LLM training — SFT, RLHF, DPO, PPO, GRPO etc.
- **When to use:** Whenever you are doing any form of LLM fine-tuning, `SFTTrainer` alone makes training dramatically simpler than raw `Trainer`
- **Sits on top of:** `transformers` + `peft` and extends `Trainer` with LLM-specific training workflows
- **Important Links:**
    - 🔗 [Documentation](https://huggingface.co/docs/trl)
    - 📄 [Paper](https://arxiv.org/abs/2204.05862)
    - 🔗 [GitHub](https://github.com/huggingface/trl)
- **Trainers in `trl`:**

| Trainer | Full Name | Description |
|---|---|---|
| `SFTTrainer` | **Supervised Fine-Tuning** Trainer | Teaches the model to follow instructions by training on (instruction, response) pairs — automatically handles dataset formatting, chat templates, response-only loss masking, and packing |
| `DPOTrainer` | **Direct Preference Optimization** Trainer | Aligns model with human preferences by training on (chosen, rejected) response pairs — simpler and lower compute than PPO as no separate reward model is needed |
| `PPOTrainer` | **Proximal Policy Optimization** Trainer | Full RLHF pipeline — uses a reward model to score responses and optimizes the model using reinforcement learning — more complex than DPO but gives finer control over alignment |
| `GRPOTrainer` | **Group Relative Policy Optimization** Trainer | Teaches the model to reason step-by-step by comparing groups of sampled responses using a reward function — used in DeepSeek-R1 |
| `RewardTrainer` | **Reward Model** Trainer | Trains a reward model on human preference data — this reward model is then used by `PPOTrainer` to generate reward signals during RLHF |

- **Example TRL Pipeline (Automated LoRA Injection using`SFTTrainer`):**
```python
# HuggingFace Transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

model = AutoModelForCausalLM.from_pretrained(...)                            # Load the pretrained model, same HuggingFace loader used in the PEFT pipeline
lora_config = LoraConfig(...)                                                # Define LoRA  adapter architecture specifying rank, lora_alpha, target_modules, droptout, task_type
trainer = SFTTrainer(model, tokenizer, dataset, lora_config, SFTConfig(...)) # Create SFTTrainer, that handles chat template handling as well
trainer.train()                                                              # Train
```

## d. **`unsloth`** (2–5× Faster Fine-Tuning)
<h3 align="center"><div class="alert alert-success" color=magenta style="margin: 20px"><b>Unsloth</b> is an open-source library that makes LoRA and QLoRA fine-tuning <b>2–5× faster</b> and uses <b>60% less VRAM</b> than the standard transformers + peft + trl stack,  by rewriting critical GPU kernels in Triton, with zero loss in accuracy. It supports <b>SFT and alignment</b> but is not designed for pre-training or continued pre-training on raw text corpora.</div></h3>


- **Problem it solves:** The standard `transformers` + `peft` + `trl` stack,  while correct and flexible, is not maximally efficient. It uses PyTorch's default attention and backward pass implementations which have room for significant optimization. Unsloth replaces these with hand-written Triton GPU kernels that are 2–5× faster
- **What it is:** An open-source library that dramatically speeds up LoRA/QLoRA fine-tuning by rewriting attention and backpropagation kernels in Triton (low-level GPU code)
    - **2–5× faster** training than standard `transformers` + `peft`
    - **60% less VRAM** — enables fine-tuning larger models on smaller GPUs
    - Drop-in replacement for `transformers` + `peft` — same API, just faster
    - Provides pre-quantized `bnb-4bit` model versions on HF Hub
    - Supports saving to GGUF format directly for local deployment
- **When to use:** When training speed and VRAM efficiency matter — especially on free Colab T4
- **Sits on top of:** all three — `transformers`, `peft`, and `trl`
- **Limitation:** Only supports specific model architectures (Llama, Mistral, Qwen, Phi, Gemma etc.) — not every model is supported
- **Important Links:**
    - 🔗 [Documentation](https://docs.unsloth.ai)
    - 🔗 [GitHub](https://github.com/unslothai/unsloth)
- **ExampleUnsloth Pipeline (Optimized Training):**
```python
model, tokenizer = FastLanguageModel.from_pretrained(...)          # Load optimized model
model = FastLanguageModel.get_peft_model(...)                      # Inject LoRA adapters, internally perform PET LoRA injection, base model freeezing and optimized LoRA kernels
trainer = SFTTrainer(model, tokenizer, dataset, SFTConfig(...))    # Create trainer (still uses TRL SFTTrainer)
trainer.train()                                                    # Same logical training loop, but using Unsloth-optimized CUDA kernels
```

# <span style='background :lightgreen' >6. [A Deep Dive into the Working of LoRA - Low-Rank Adaptation](https://arxiv.org/abs/2106.09685)</span>

## a. LLaMA-7B Model Weights/Parameters
- A 7B model (like LLaMA-7B) has a hidden size of 4096, having 32 layers, a vocab of 32768 and MLP expansion of 11008 (SwiGLU).
- It uses RoPE (parameter-free) for positional-encoding and uses bias-free linear layers.
- LLaMA uses RMSNorm instead of LayerNorm for efficiency and stability. It uses a vector of size `hidden_dim` (e.g., 4096) that performs element-wise scaling after normalization. (scale-invariant normalization + learnable feature-wise reweighting).
- LLaMA-7B has many different weight matrices of varying shapes as shown below:
```
    - Embedding matrix:                    32768 × 4096  = 134,217,728 parameters × 1                  = 134,217,728
    - Attention Q projection:              4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - Attention K projection:              4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - Attention V projection:              4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - Attention output projection:         4096  × 4096  = 16,777,216  parameters × 32 layers          = 536,870,912
    - MLP gate projection (SwiGLU):        4096  × 11008 = 45,088,768  parameters × 32 layers          = 1,442,840,576
    - MLP up projection:                   4096  × 11008 = 45,088,768  parameters × 32 layers          = 1,442,840,576
    - MLP down projection:                 11008 × 4096  = 45,088,768  parameters × 32 layers          = 1,442,840,576
    - RMSNorm (2 per layer):               (2 x 4096)    = 8192        parameters × 32 layers          = 262,144 + 4096 (final RMSNorm)
    - LM Head:                             32768 × 4096  = 134,217,728 parameters × 1 (tied to EM)     = 0
```
- This sums up to:
```
Embedding:          134,217,728
Attention:        2,147,483,648
MLP:              4,328,521,728
RMSNorm:                266,240
--------------------------------
TOTAL ≈          6,610,489,344 ≈ 6.61B
```
>- The remaining ~0.39B gap is due to rounding and minor architectural variations.

## b. The Problem — Why Full Fine-Tuning is Impractical
- A 7B model in fp16 requires **14 GB** just to store the weights at inference time
- Fine-tuning needs approximately 8× more memory than inference (a model that runs in 14 GB needs 112 GB to fine-tune!)
- The complete  memory breakdown for full fine-tuning a 7B model looks like this:
- **FULL FINE-TUNING — 7B Model Memory Breakdown with Adam (Adaptive Moment Estimation) Optimizer:**
```
─────────────────────────────────────────────────────────────────
Component               │ Precision │ Bytes/param │ Total Memory
────────────────────────┼───────────┼─────────────┼─────────────
1. Model Weights        │ fp16      │ 2 bytes     │  14.0 GB
2. Gradients            │ fp16      │ 2 bytes     │  14.0 GB
3. fp32 Master Weights  │ fp32      │ 4 bytes     │  28.0 GB
4. Adam Momentum (m)    │ fp32      │ 4 bytes     │  28.0 GB
5. Adam Variance (v)    │ fp32      │ 4 bytes     │  28.0 GB
─────────────────────────────────────────────────────────────────
TOTAL                                             │ 112.0 GB
─────────────────────────────────────────────────────────────────
```
>- Free tier of Google's Colab provides you a T4 with 16GB of VRAM, which is OK for inference, but not for fine-tuning of a 7B model
>- Colab Pro+ gives you access to an A100 GPU with either 40GB or 80GB VRAM — both are still far short of the 112 GB required
>- Cloud platforms like AWS or Google Cloud, renting an 8× A100 instance costs roughly `$29 – $33` per hour on-demand — and that is just for the compute, before storage and data transfer fees
>- Purchasing a single A100 GPU costs approximately `$8000 – $15000`, depending on VRAM variant (40GB vs 80GB)


#### 💡 Understanding Why Each Component Exists
- Before understanding how LoRA solves the problem, it is important to understand WHY each of the a/m 5 memory components are needed in the first place:
    - **Model Weights (14 GB):** The actual 7B parameters are stored in fp16 that define every computation the model performs during the forward pass.
    - **Gradients (14 GB):** During the backward pass, every trainable weight needs a gradient, a number telling Adam *"update this weight by this amount"*. In full fine-tuning: `7B × 2 bytes (fp16) = 14 GB` just for gradients
    - **fp32 Master Weights (28 GB):** Mixed precision training keeps a full fp32 copy of every weight alongside the fp16 working copy. This is because fp16 cannot accurately represent very small weight updates (e.g., 0.000001), as it rounds them to zero and training stalls. So Adam applies gradient updates to the fp32 master copy, then casts back to fp16 for the forward pass
    - **Adam Momentum (28 GB):** Momentum in Adam controls the *direction of the movement*. Instead of only looking at the current gradient, it keeps a running average of past gradients: `m = 0.9 × m_previous + 0.1 × current_gradient`. This smooths the learning path, reduces zig-zag movement, and helps the model reach the minimum faster and more steadily.
    - **Adam Variance (28 GB):** Variance in Adam controls the *effective step size (learning rate)* for each parameter. The Adam optimizer tracks the exponentially weighted average of squared gradients: `v = 0.999 × v_previous + 0.001 × gradient²`. If a weight’s gradients are large or noisy, updates become smaller. If a weight’s gradients are small and stable, updates can be larger.

> 💡 **Why fp32 for optimizer states?** Adam's momentum and variance are running averages accumulated over many steps. If stored in fp16, small gradient updates get rounded to zero and training becomes unstable. fp32 is mandatory here for numerical stability.


## c. What is Rank of a Matrix?
- The maximum number of `linearly independent` columns (or rows ) of a matrix is called its rank. (Two columns are linearly dependent if you can add+scale one column to make the other)
- In other words: rank counts how many rows/columns cannot be derived from the other rows/columns by simple multiplication or addition
- Key rules to remember:
    - The rank of an $m×n$ matrix is always $\leq \min(m, n)$
    - A matrix with no elements has rank $=0$
    - Any matrix with at least one non-zero element has rank $\geq 1$
- Linear Algebra techniques to find the rank of a matrix are:
    - Row Echelon Form (REF) via Gaussian Elimination
    - Reduced Row Echelon Form (RREF) via Gauss–Jordan Elimination
    - Column Echelon Form (CEF) via Column Reduction

**Example 1:** Consider the following hypothetical $4 \times 5$ matrix (say containing weights of a neural network model). Can you identify its rank using pen and paper?

$\hspace{3 cm}W_{4,5} = \begin{bmatrix} 
       1 & 2 & 5 & 3 & 4\\ 
       3 & 3 & 9 & 6 & 9\\
       2 & 3 & 8 & 5 & 7\\
       4 & 1 & 6 & 5 & 9
\end{bmatrix}$

> 💡 **Hint:** Look carefully at the columns. Can you express any column as a combination of the other columns? Try to find the pattern focusing on columns 1 and 2 first.

<details>
<summary>✅ Click to reveal the answer</summary>
<br>

**Try expressing col3, col4, col5 using col1 and col2:**
- $col_3 = col_1 + 2 \times col_2$
- $col_4 = col_1 + col_2$
- $col_5 = 2 \times col_1 + col_2$

**Conclusion:**
- $col_3$, $col_4$, $col_5$ are **linearly dependent** on $col_1$ and $col_2$
- Only $col_1$ and $col_2$ are truly **linearly independent**
- Therefore: $\text{rank}(W) = \mathbf{2}$
</details>

In [1]:
import numpy as np
W = np.array([
    [1, 2, 5, 3, 4],
    [3, 3, 9, 6, 9],
    [2, 3, 8, 5, 7],
    [4, 1, 6, 5, 9]
])
print(W)
print(f"Maximum possible rank   : min({W.shape[0]}, {W.shape[1]}) = {min(W.shape)}")
print(f"Actual rank of W        : {np.linalg.matrix_rank(W)}") # Use SVD, which is the most numerically stable out of a/m techniques

[[1 2 5 3 4]
 [3 3 9 6 9]
 [2 3 8 5 7]
 [4 1 6 5 9]]
Maximum possible rank   : min(4, 5) = 4
Actual rank of W        : 2


## d. What is Low-Rank Decomposition?

<h3 align="center"><div class="alert alert-success" style="margin: 20px">
<b>Low-Rank Decomposition</b> is the process of approximating or exactly representing a large matrix as a product of two smaller matrices $A_{r \times d_{in}}$ and $B_{d_{out} \times r}$
</div></h3>

- We all have done factorization or prime factorization of integers:
    - $ 18 = 9 * 2$
    - $ 18 = 3 * 3 * 2$
    - $ 18 = 6 * 3$
    - $ 18 = 2 * 3 * 3$
- Similar to factorizing an integer into its prime factors, Matrix decomposition is the process of **reducing a matrix to its constituent parts** in order to make certain subsequent matrix calculations simpler. 
- **Matrix Decomposition Techniques:**
    - LU Decomposition: Factorizes a matrix into a lower triangular matrix (L) and an upper triangular matrix (U) to simplify solving linear systems.
    - LR Decomposition: A variant of LU decomposition where the matrix is expressed as a product of lower (L) and upper (R) triangular matrices.
    - QR Decomposition: Factorizes a matrix into an orthogonal matrix (Q) and an upper triangular matrix (R) to reveal independent directions in data.
    - Eigen Decomposition: Decomposes a square matrix into eigenvectors and eigenvalues to identify directions that remain unchanged under transformation.
    - Singular Value Decomposition (SVD): Decomposes any matrix into orthogonal components and singular values to capture the most important structure in the data.
- **Common Use Cases of Low-Rank Matrix Decomposition:**
    - Dimensionality Reduction: Techniques like PCA (based on SVD) reduce the number of features while keeping most of the important information (variance) in the data.
    - Data Compression: Large datasets (e.g., images) can be approximated using a low-rank version by keeping only the most important components. This significantly reduces storage while preserving key information.
    - Image Processing: Low-rank approximation helps remove noise and redundancy in images. It is commonly used in image denoising, background removal, and face recognition (Eigenfaces).
    - Fine-Tuning of LLMs (LoRA): Low-rank decomposition is used in techniques like Low-Rank Adaptation (LoRA) to efficiently fine-tune large language models. Instead of updating the entire model, only small low-rank matrices are trained, which reduces memory usage and computation cost.

>- [Applied Linear Algebra - I](https://www.youtube.com/watch?v=CLGbFiZStJU&list=PL7B2bn3G_wfCMuly7_NzKfgxon1wPUe8a&index=2) 
>- [Applied Linear Algebra - II](https://www.youtube.com/watch?v=Z2WDUUHkof8&list=PL7B2bn3G_wfCMuly7_NzKfgxon1wPUe8a&index=1) 

**Example 2:** 
- Given the following weight matrix $W_{4 \times 5}$, which has a rank of 2.
- This means \( W \) can be **exactly decomposed** into two smaller matrices $A_{2 \times 5}$ and $B_{4 \times 2}$ such that $B \times A = W$, where:

$A \in R^{2 \times d_{in}} = R^{2 \times 5} \quad   d_{in}  = 5 \quad \text{(size of input vector i.e., number of columns of W)}$

$B \in R^{d_{out} \times 2} = R^{4 \times 2} \quad  d_{out} = 4 \quad \text{(size of output vector (number of rows of W)}$

$W_{4 \times 5} = \begin{bmatrix} 
1 & 2 & 5 & 3 & 4\\ 
3 & 3 & 9 & 6 & 9\\
2 & 3 & 8 & 5 & 7\\
4 & 1 & 6 & 5 & 9
\end{bmatrix} = B_{4 \times 2} \times A_{2 \times 5}$

> 💡 **Hint:** From Example 1, we know that $col_1$ and $col_2$ of $W$ are the only two independent columns.  
> Use these columns to construct matrix $B$ (basis matrix). Then express each column of $W$ as a linear combination of these basis columns to construct matrix $A$, such that $B \times A$ reconstructs $W$.


<details>
<summary>✅ Click to reveal the answer</summary>
<br>

- **Step 1 — Build matrix $B$ using the two independent columns of $W$:**
$B_{4 \times 2} = \begin{bmatrix} 1 & 2\\ 3 & 3\\ 2 & 3\\ 4 & 1 \end{bmatrix}$
- **Step 2 — Build matrix $A$ using the coefficients found in Example 1:**
From Example 1, we know how each column of $W$ is built from $col_1$ and $col_2$:
- $col_1 = 1 \times col_1 + 0 \times col_2$
- $col_2 = 0 \times col_1 + 1 \times col_2$
- $col_3 = 1 \times col_1 + 2 \times col_2$
- $col_4 = 1 \times col_1 + 1 \times col_2$
- $col_5 = 2 \times col_1 + 1 \times col_2$
- These coefficients become the **columns of $A$**:
$A_{2 \times 5} = \begin{bmatrix} 1 & 0 & 1 & 1 & 2\\ 0 & 1 & 2 & 1 & 1 \end{bmatrix}$
- **Step 3 — Verify that $B \times A = W$:**
$B \times A = \begin{bmatrix} 1 & 2\\ 3 & 3\\ 2 & 3\\ 4 & 1 \end{bmatrix} \times \begin{bmatrix} 1 & 0 & 1 & 1 & 2\\ 0 & 1 & 2 & 1 & 1 \end{bmatrix} = \begin{bmatrix} 1 & 2 & 5 & 3 & 4\\ 3 & 3 & 9 & 6 & 9\\ 2 & 3 & 8 & 5 & 7\\ 4 & 1 & 6 & 5 & 9 \end{bmatrix} = W \quad$
- **Conclusion:**
- $W_{4 \times 5}$ (20 values) is **exactly reproduced** by $B_{4 \times 2} \times A_{2 \times 5}$ (8 + 10 = **18 values**)
- $W_{31623 \times 31623}$ (~1 billion or $10^9$ values) is **exactly reproduced** by $B_{31623 \times 16} \times A_{16 \times 31623}$ (505,968 + 505,968 = **1,011,936 values**) — a saving of over **98.8%** 🔥
- This is the essence of **low-rank decomposition** — a large matrix represented by two much smaller matrices
- In LoRA, instead of updating the full $W_{m \times n}$ we learn small matrices $A$ and $B$ where the rank $r \ll \min(m, n)$
</details>

## e. Why Rank Matters for LoRA?
- In one layer of LLaMA-7B a single **Q- projection weight matrix** has (4096 × 4096) 16,777,216 weights, that are stored and full rank would mean rank = 4096
- The LoRA paper says that when you fine-tune a large model on a specific task, the *change* in weights (ΔW) that the model needs to learn has a very **low intrinsic rank**, i.e., the weight update matrix can be accurately approximated using two much smaller matrices
- In practice: effective rank of ΔW ≈ 1 to 64 out of a possible maximum rank of 4096 
- In other words: **you do not need to update all 7B parameters to teach the model a new task — the actual new information needed is much smaller**
- Think of weight updates as DIRECTIONS in space:
    - r = 1  → model can only learn changes along 1 direction. Very limited — simple tasks only
    - r = 4  → model can learn changes along 4 directions. Good for simple fine-tuning tasks
    - r = 16 → model can learn changes along 16 directions. Most commonly used value for rank
    - r = 64 → model can learn changes along 64 directions. More expressive but uses more memory. Needed for complex tasks with diverse data
    - r = 4096 → full rank — every possible direction. This is full fine-tuning — maximum expressiveness

## f. How LoRA Exploits The Low-Rank Decomposition?
```
Full Fine-Tuning (Full rank update (wasteful):            Low-rank decomposition (LoRA's approach):
───────────────────────────────────────────────────────────────────────────────────────────────────────
W_new = W_original + ΔW                                   W_new = W_original + B × A

Where:                                                    Where:
  W_original → frozen (never updated)                            W_original → frozen (never updated)
  ΔW         → full size matrix (e.g., 4096 × 4096)              ΔW        → B × A → two tiny matrices
             = 16,777,216 params                                 A is (r x d_in) and B is (d_out x r)
  Full rank would mean, rank = 4096                              If r = 16: (controls how many independent "directions of change" we allow the model to learn
                                                                     B×A = (4096×16) + (16×4096) = 131,072 params (128× fewer params!)
```

## g. Step by Step Working of LoRA
- **[TinyLlama-1.1B-Chat (Instruction-tuned)](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0)** is a decoder-only Transformer language model based on the LLaMA architecture. It contains 22 transformer decoder layers, a 2048 hidden size, and uses 32 attention heads. It uses Grouped-Query Attention (GQA) with 32 query heads and 4 key-value heads, along with RMSNorm for normalization, SwiGLU feed-forward networks, Rotary Positional Embeddings (RoPE), and a 2048-token context window.
```
Each Decoder Layer
├── Self-Attention
│   ├── Q  (Query projection matrix)   ← W_q    shape: 2048 × 2048   ( 32 heads   × 64 head_dim)
│   ├── K  (Key projection matrix)     ← W_k    shape: 2048 × 256    ( 4 KV heads × 64 head_dim)
│   ├── V  (Value projection matrix)   ← W_v    shape: 2048 × 256    ( 4 KV heads × 64 head_dim)
│   └── O  (Output projection matrix)  ← W_o    shape: 2048 × 2048
└── Feed-Forward Network (SwiGLU)
    ├── Gate matrix         ← W_gate  shape: 2048 × 5632
    ├── Up matrix           ← W_up    shape: 2048 × 5632
    └── Down matrix         ← W_down  shape: 5632 × 2048
```
- **Note:**
>- In the original transformer architecture, each attention head has its own Q, K and V.
>- Grouped-Query Attention (GQA) is a variant of multi-head attention where multiple query heads share a smaller number of key and value heads to reduce computation and memory while maintaining performance.
>- In TinyLlama, Q has 32 heads but K and V only have 4 heads. This means W_q and W_o are 2048 × 2048 while W_k and W_v are only 2048 × 256.
>- To keep things simple in the steps below, we will only inject LoRA into W_q and W_o (both 2048 × 2048) and avoid W_k and W_v.

- **Step 1 — Freeze the entire TinyLlama-1.1B base model:**
    - All original pretrained weights across all **22 decoder layers** are locked and will never be updated.
    - This preserves all the general language knowledge learned during TinyLlama's pretraining on **3 trillion tokens**
    - TinyLlama's original weight matrices (e.g. `q_proj`, `o_proj` of shape `2048 × 2048`) become pure **read-only** during fine-tuning<br><br>
- **Step 2 — Inject LoRA adapter matrices A and B into selected layers:**
    - For each of the **22 decoder layers**, two pairs of small matrices are inserted alongside the selected projection matrices (typically `q_proj` and `o_proj`):
        - **Matrix A** → shape `(16 x 2048)` — initialized with random Gaussian values
        - **Matrix B** → shape `(2048 x 16)` — initialized with **zeros**
    - Starting $B$ at zero ensures $\Delta W = B \times A = 0$ at the start — meaning fine-tuning begins from **exactly the same behavior** as the original TinyLlama model with no sudden disruption
    - Total trainable parameters: 22 layers $\times$ 2 projections $\times (2048 \times 16 + 16 \times 2048) = 2883584 \approx$ **~2.9M** (out of 1.1B,  only ~0.26% of the model 🔥 <br><br>
- **Step 3 — Train ONLY matrices A and B:**
    - During each forward pass through TinyLlama, the output of each adapted layer becomes:<br>
$\text{output} = \underbrace{W_{\text{original}} \times x}_{\text{frozen path}} + \underbrace{(B \times A) \times x \times \frac{\alpha}{r}}_{\text{LoRA adapter path}}$
    - Only $A$ and $B$ receive gradients — the frozen `q_proj` and `o_proj` weight matrices of TinyLlama **never change**
    - $\alpha$ (`lora_alpha`) is a scaling factor controlling how strongly the adapter influences TinyLlama's output — typically set to `2 x r`
    - $r$ (`lora_rank`) controls the rank of the decomposition — typically set to `8` or `16` <br><br>
- **Step 4 — Merge adapters for inference (optional):**
    - After fine-tuning is complete, the adapter can be merged back into TinyLlama's original weights:
$W_{\text{merged}} = W_{\text{original}} + \frac{\alpha}{r} . B \times A$
    - The result is a single standard TinyLlama-style weight matrix with **zero inference overhead** — no adapter logic remaining at runtime
    - Alternatively, keep $A$ and $B$ **separate and swappable** — allowing you to maintain one frozen TinyLlama base and **hot-swap multiple LoRA adapters** for different tasks (e.g. one for coding, one for medical QA, one for Urdu language) without reloading the entire model

## h. Key LoRA Hyperparameters
- **`target_modules` — Which Layers to Apply LoRA To**
    - Typically applied to attention projection layers:
        `q_proj`, `k_proj`, `v_proj`, `o_proj`
    - Can also include MLP layers: `gate_proj`, `up_proj`, `down_proj`
    - More modules → better results but more memory
- **`r` — Rank:**
    - Controls the size of adapter matrices A and B
    - Smaller r → fewer params → faster, less memory, less expressive
    - Larger r → more params → slower, more memory, more expressive
    - Typical values: `4, 8, 16, 32`
- **`lora_alpha` — Scaling Factor**
    - Controls how much the adapter influences the output via `α/r`
    - Common practice: set `lora_alpha = 2 × r` (e.g., if r=16 → alpha=32)
- **`lora_dropout`**
    - Its a technique that try to ensure that your neural network doesn't overfit to the data.
    - Typical value is `0.1`, that means 10% of the neurons (linear regression units) inside a deep neural network get wiped out or zeroed out when you are doing your training and a different 10% each time.
    - Unsloth doesn't support this hyperparameter, so we set it to `0` for Unsloth

## i. LoRA Architecture Diagram
```
  Input x
     │
     ▼
┌────────────────────────────────────────────┐
│          Pretrained Weight W               │
│         (FROZEN — never updated)           │
│              d × d matrix                  │
└────────────────────────────────────────────┘
     │                    │
     │              ┌─────▼──────┐
     │              │  Matrix A  │  ← trained │ r × d │ initialized: random
     │              └─────┬──────┘
     │                    │  (low-rank bottleneck)
     │              ┌─────▼──────┐
     │              │  Matrix B  │  ← trained │ d × r │ initialized: zeros
     │              └─────┬──────┘
     │                    │  × (α / r)   scaling
     ▼                    ▼
  W · x        +        B · A · x
     │                     │
     └──────────┬──────────┘
                ▼
            Output h
   (pretrained + adapter contribution)
```

## j. How LoRA Solves the 112 GB Problem

<h3 align="center"><div class="alert alert-success" style="margin: 20px">LoRA does not shrink the model itself—it reduces what needs to be trained and stored during training.</div></h3>


- LoRA attacks 4 out of 5 memory components — the only component it cannot reduce is the model weights themselves (since the base model must still be loaded in fp16 for the forward pass
    - **Base weights (14 GB → 14 GB — unchanged:** LoRA freezes the base model but still needs to load all 7B weights in fp16 for the forward pass
    - **Gradients (fp16):** Gradients are computed only for small adapter matrices
    - **Master weights (fp32):** Master weights for small adapter matrices
    - **Adam momentum (fp32):** Adam momentum for small adapter matrices
    - **Adam variance (fp32):** Adam momentum for small adapter matrices

⚠️ **LoRA alone brings 112 GB down to ~14.5 GB (87% reduction)**

>- A massive improvement, but still tight for a free Colab T4 GPU (15 GB VRAM).
>- The remaining bottleneck is the **14 GB of fp16 model weights** which LoRA cannot reduce — that is exactly the problem that **QLoRA solves** by loading the base model in NF4 4-bit (0.5 bytes/param) instead of fp16 (2 bytes/param), bringing the total down to **~4 GB**.

# <span style='background :lightgreen' >7. Example Questions to Check your Understanding about LoRA</span>

- You are fine-tuning a transformer model that has a single attention layer with the following weight matrices:

```
Query projection  : W_Q  shape = (2048 × 2048)
Key projection    : W_K  shape = (2048 × 2048)
Value projection  : W_V  shape = (2048 × 2048)
Output projection : W_O  shape = (2048 × 2048)
```
- You decide to apply LoRA with rank **r = 8** to all four matrices.
- The model weights are in **fp16** (2 bytes/param), gradients in **fp32**,  and trained with **Adam optimizer** (momentum and variance stored in fp32). Assume **no quantization** (plain LoRA, not QLoRA).
---

- **(a)** How many parameters does ONE weight matrix (e.g., W_Q) have?
- **(b)** How many total parameters are there in all four matrices combined?
- **(c)** What is the shape of LoRA matrices and total trainable parameters across all four matrices (Q, K, V, O)?
- **(d)** What percentage of the original 4-matrix parameters are actually trained via LoRA?
- **(e)** How much memory do the 4 frozen base weight matrices  consume in fp16?
- **(f)** How much memory do ALL LoRA adapter weights consume in fp16?
- **(g)** Calculate and fill in the following comparison table:

```
Component              │Full Fine-Tuning │ LoRA (r=8) │ Saving
───────────────────────┼─────────────────┼────────────┼───────
Base weights (fp16)    │                 │            │
Gradients (fp16)       │                 │            │
fp32 master weights    │                 │            │
Adam momentum (fp32)   │                 │            │
Adam variance (fp32)   │                 │            │
───────────────────────┼─────────────────┼────────────┼───────
TOTAL                  │                 │            │
```

- **(h)** At initialization, Matrix A is initialized with random Gaussian values and Matrix B is initialized with all zeros. What is the value of the LoRA update `ΔW = B × A` at initialization? Why is this initialization strategy important?
- **(i)** After LoRA training is complete, you have two options for inference:
    - **Option 1:** Keep base model + load LoRA adapters separately
    - **Option 2:** Merge adapters into base weights: `W_new = W_base + B × A`
    - For each option state:
        - How much memory is needed? (base weights + adapter weights if any)
        - Is there any computational overhead at inference time?
        - When would you prefer each option?

<details>
<summary>💡 <b>Click to reveal answer</b></summary>

```
Given:
  Matrix shape       = 2048 × 2048
  Number of matrices = 4  (W_Q, W_K, W_V, W_O)
  LoRA rank          = r = 8
  Base model         = fp16 = 2 bytes/param
  fp32               = 4 bytes/param
  Adam optimizer     = momentum + variance in fp32


(a) Parameters in ONE weight matrix:
    W_Q = 2048 × 2048 = 4,194,304 params

(b) Total parameters in all four matrices:
    4 × 4,194,304 = 16,777,216 params

(c) LoRA matrix shapes and trainable parameter count:
    For each weight matrix of shape (2048 × 2048):
      A shape = (r    × d_in)  = (8    × 2048) =  16,384 params
      B shape = (d_out × r   ) = (2048 × 8   ) =  16,384 params
      LoRA params per matrix   =  16,384 + 16,384 = 32,768 params

    Across all four matrices (W_Q, W_K, W_V, W_O):
      Total LoRA trainable params = 4 × 32,768 = 131,072 params

(d) Percentage of original parameters trained via LoRA:
    = (131,072 / 16,777,216) × 100 = 0.78125%
                        → Using LoRA, we train around 0.78% of the original parameters!

(e) Memory for 4 frozen base weight matrices in fp16:
    = 16,777,216 × 2 bytes = 32 MB

(f) Memory for ALL LoRA adapter weights in fp16:
    = 131,072 × 2 bytes =  0.25 MB

    
(g) Comparison Table calculations:

  ── Full Fine-Tuning (all 16,777,216 params trained) ──────────
  Base weights (fp16)  = 16,777,216 × 2 = 33,554,432 bytes = 32.0 MB
  Gradients (fp16)     = 16,777,216 × 2 = 33,554,432 bytes = 32.0 MB
  fp32 master weights  = 16,777,216 × 4 = 67,108,864 bytes = 64.0 MB
  Adam momentum (fp32) = 16,777,216 × 4 = 67,108,864 bytes = 64.0 MB
  Adam variance (fp32) = 16,777,216 × 4 = 67,108,864 bytes = 64.0 MB
  TOTAL = 32.0 + 32.0 + 64.0 + 64.0 + 64.0 = 256.0 MB

  ── LoRA r=8 (only 131,072 params trained, base frozen) ───────
  Base weights (fp16)    = 16,777,216 × 2 = 32.0 MB  ← still loaded, just frozen
  Gradients (fp16)       =    131,072 × 2 =  0.25 MB ← only for LoRA params
  Master weights (fp32)  =    131,072 × 4 =  0.5 MB  ← only for LoRA params
  Adam momentum (fp32)   =    131,072 × 4 =  0.5 MB  ← only for LoRA params
  Adam variance (fp32)   =    131,072 × 4 =  0.5 MB  ← only for LoRA params
  TOTAL = 32.0 + 0.25 + 0.5 + 0.5 + 0.5 = 33.75 MB

  ── Savings ───────────────────────────────────────────────────
  Base weights     : 32.0 - 32.0  =  0 MB    (0.0%  — always required)
  Gradients        : 32.0 - 0.25  = 31.75 MB (99.2% reduction)
  Master weights   : 64.0 - 0.5   = 63.5 MB  (99.2% reduction)
  Adam momentum    : 64.0 - 0.5   = 63.5 MB  (99.2% reduction)
  Adam variance    : 64.0 - 0.5   = 63.5 MB  (99.2% reduction)
  TOTAL saving     : 256.0 - 33.75 = 222.25 MB (86.8% reduction)

┌───────────────────────┬─────────────────┬────────────┬──────────────────────┐
│ Component             │Full Fine-Tuning │ LoRA (r=8) │ Saving               │
├───────────────────────┼─────────────────┼────────────┼──────────────────────┤
│ Base weights (fp16)   │     32.0 MB     │   32.0 MB  │  0 MB (always needed)│
│ Gradients (fp16)      │     32.0 MB     │    0.25 MB │ 31.75 MB  (99.2%)    │
│ fp32 master weights   │     64.0 MB     │    0.5 MB  │ 63.5 MB   (99.2%)    │
│ Adam momentum (fp32)  │     64.0 MB     │    0.5 MB  │ 63.5 MB   (99.2%)    │
│ Adam variance (fp32)  │     64.0 MB     │    0.5 MB  │ 63.5 MB   (99.2%)    │
├───────────────────────┼─────────────────┼────────────┼──────────────────────┤
│ TOTAL                 │    256.0 MB     │   33.75 MB │ 222.25 MB (86.8%)    │
└───────────────────────┴─────────────────┴────────────┴──────────────────────┘

(h) LoRA update at initialization:
    A ← random Gaussian (small non-zero values)
    B ← all zeros

    ΔW = B × A = 0 × A = 0  (zero matrix, all entries = 0)

    Why this initialization strategy is important:
    - At the start of training, ΔW = 0 means the adapted model
      is IDENTICAL to the original pretrained model
    - This ensures training begins from a stable, known starting
      point — the pretrained checkpoint — without any random
      disruption to the model's existing knowledge
    - If B were also initialized randomly, ΔW would be a random
      matrix that would immediately corrupt the pretrained
      representations before any useful learning has occurred
    - The zero initialization preserves the pretrained model's
      capabilities at step 0 and lets the adapter learn only
      the task-specific delta from that clean starting point

(i) Inference Options:

  ┌──────────────────────┬────────────────────────────────────────────────────┐
  │ Option 1             │ Keep base model + LoRA adapters separately         │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Memory needed        │ Base weights : 32.0 MB (fp16)                      │
  │                      │ LoRA adapter :  0.25 MB (fp16)                     │
  │                      │ TOTAL        : 32.25 MB                            │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Computational        │ YES — at every forward pass must compute:          │
  │ overhead             │   output = W_base.x + (B.A.x) × scaling_factor     │
  │                      │ Two extra small matrix multiplications per layer   │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ When to prefer       │ Serving MULTIPLE fine-tuned tasks from the SAME    │
  │                      │    base model — swap only the 0.25 MB adapter      │
  │                      │    between tasks without reloading the base model  │
  │                      │ Storage-constrained deployments with many tasks    │
  │                      │ During active experimentation and iteration        │
  └──────────────────────┴────────────────────────────────────────────────────┘

  ┌──────────────────────┬────────────────────────────────────────────────────┐
  │ Option 2             │ Merge adapters: W_new = W_base + B × A             │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Memory needed        │ Merged weights : 32.0 MB (fp16)                    │
  │                      │ LoRA adapter   :  0 MB  (discarded after merge)    │
  │                      │ TOTAL          : 32.0 MB                           │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ Computational        │ NO — merging is a ONE-TIME operation before        │
  │ overhead             │ deployment. W_new is just a regular weight matrix. │
  │                      │ Inference is identical to a standard model         │
  ├──────────────────────┼────────────────────────────────────────────────────┤
  │ When to prefer       │ Deploying a SINGLE fine-tuned model to production  │
  │                      │    where inference latency matters                 │
  │                      │ When you want zero overhead and the simplest       │
  │                      │    possible inference pipeline                     │
  │                      │ When the base model is not shared across tasks     │
  └──────────────────────┴────────────────────────────────────────────────────┘

  Key insight: merging costs 0 extra memory (W_new is the same
  shape as W_base) and eliminates all inference overhead —
  the only trade-off is losing the ability to swap adapters
  without re-merging.
```

</details>